import matplotlib.pyplot as plt
# Protocole commun d'?valuation Elliptic

Ce notebook compare six mod?les avec le m?me d?coupage temporel, les m?mes donn?es pr?par?es et les m?mes r?gles d'?valuation. Il n'est pas ex?cut? ? sa g?n?ration. Depuis `notebooks/`, lancez-le avec la commande donn?e dans le README.

Les transactions `unknown` restent des n?uds du graphe, mais ne contribuent ni ? l'entra?nement supervis? ni aux m?triques. Chaque seuil est s?lectionn? sur la validation et fig? avant le test.

In [ ]:
# Configuration centrale : passer ? SEEDS = [42] et GCN_EPOCHS = 20 pour un essai rapide.
from pathlib import Path
import sys
import numpy as np

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))

from src.data import load_data

SEEDS = [42, 43, 44, 45, 46]
GCN_EPOCHS = 600
MLP_EPOCHS = 200
MODELES = ["lr", "rf", "mlp", "xgboost"]

# Chargement puis assertions des dimensions et effectifs Elliptic attendus.
data = load_data()
assert data.features.shape == (203_769, 167), data.features.shape
assert data.edges.shape == (234_355, 2), data.edges.shape
assert len(data.classes) == 203_769, len(data.classes)
counts = data.classes["class"].astype(str).value_counts().to_dict()
assert counts.get("1", 0) == 4_545, counts
assert counts.get("2", 0) == 42_019, counts
assert counts.get("unknown", 0) == 157_205, counts
assert int(np.sum(data.y[data.test_mask] == 1)) == 1_083
assert not np.any(data.train_mask & data.val_mask)
assert not np.any(data.train_mask & data.test_mask)
assert not np.any(data.val_mask & data.test_mask)
assert not np.any((data.train_mask | data.val_mask | data.test_mask) & (data.y < 0))
print(f"Donn?es v?rifi?es : {data.features.shape[0]:,} n?uds, {data.edges.shape[0]:,} ar?tes")
print(f"Classes : illicites={counts['1']:,}, licites={counts['2']:,}, unknown={counts['unknown']:,}")
print(f"Support illicite test : {int(np.sum(data.y[data.test_mask] == 1)):,}")

## Ex?cution des mod?les

Chaque mod?le est entra?n? sur les labels connus des pas 1?24. Le scaler a ?t? ajust? sur les lignes des pas 1?24 uniquement. Le GCN et le Skip-GCN voient le graphe complet ; aucune ar?te ne relie deux pas diff?rents. Leur validation retient l'?poque ? meilleure PR-AUC, sans consulter les labels du test pour choisir le seuil ou l'?poque.

In [ ]:
from time import perf_counter
import pandas as pd
from src.models import run_model
from src.metrics import f1_by_time_step

runs = []
par_pas = []
for nom in MODELES:
    for seed in SEEDS:
        debut = perf_counter()
        resultat = run_model(
            nom,
            data,
            seed,
            mlp_epochs=MLP_EPOCHS,
            gcn_epochs=GCN_EPOCHS,
        )
        duree = perf_counter() - debut
        for partition in ("validation", "test"):
            ligne = {
                "modele": resultat["modele"],
                "seed": seed,
                "partition": partition,
                "seuil_validation": resultat["threshold"],
                "duree_secondes": duree,
                "meilleure_epoque": resultat["best_epoch"],
                **resultat[partition],
            }
            runs.append(ligne)
        details = f1_by_time_step(
            resultat["y_test"],
            resultat["probabilities_test"],
            resultat["time_steps_test"],
            resultat["threshold"],
        )
        details.insert(0, "seed", seed)
        details.insert(0, "modele", resultat["modele"])
        par_pas.append(details)
        print(f"{resultat['modele']} | seed={seed} | {duree:.1f} s | seuil validation={resultat['threshold']:.4f}")

runs_df = pd.DataFrame(runs)
par_pas_df = pd.concat(par_pas, ignore_index=True)
runs_df.head()

## R?sum? : moyenne ? ?cart-type

Les m?triques portent uniquement sur les transactions ?tiquet?es. La PR-AUC est calcul?e ? partir des probabilit?s ; pr?cision, rappel et F1 utilisent le seuil s?lectionn? sur la validation. L'accuracy n'est pas rapport?e.

In [ ]:
from src.metrics import aggregate_metrics

resume = aggregate_metrics(runs_df)
resume

## F1 du test par pas de temps

Le support indique le nombre r?el de transactions illicites de chaque pas. Les moyennes et ?carts-types sont calcul?s sur les seeds ; les pas ? faible support doivent ?tre interpr?t?s avec prudence.

In [ ]:
import matplotlib.pyplot as plt

pas_resume = (
    par_pas_df.groupby(["modele", "time_step"], as_index=False)
    .agg(f1_moyen=("f1", "mean"), f1_ecart_type=("f1", "std"), support_illicite=("support_illicite", "mean"))
)
fig, ax = plt.subplots(figsize=(12, 6))
for modele, groupe in pas_resume.groupby("modele"):
    ax.plot(groupe["time_step"], groupe["f1_moyen"], marker="o", label=modele)
ax.set(title="F1 illicite du test par pas de temps", xlabel="Pas de temps", ylabel="F1")
ax.set_xticks(range(35, 50))
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

# Tableau des supports et F1 par mod?le et pas (moyenne sur les seeds).
pas_resume

## Sauvegarde

Le CSV contient les m?triques brutes par seed et partition, puis les F1 par pas de temps et seed afin de conserver les supports et la dispersion.

In [ ]:
from pathlib import Path

sortie = RACINE / "results" / "protocole_commun.csv"
sortie.parent.mkdir(parents=True, exist_ok=True)
resume_csv = resume.copy()
resume_csv.insert(0, "type_ligne", "resume_moyenne_ecart_type")
metriques_csv = runs_df.copy()
metriques_csv.insert(0, "type_ligne", "metrique_par_seed")
pas_csv = par_pas_df.copy()
pas_csv.insert(0, "type_ligne", "f1_par_pas_et_seed")
colonnes = sorted(set(resume_csv.columns) | set(metriques_csv.columns) | set(pas_csv.columns))
resultats_csv = pd.concat(
    [
        resume_csv.reindex(columns=colonnes),
        metriques_csv.reindex(columns=colonnes),
        pas_csv.reindex(columns=colonnes),
    ],
    ignore_index=True,
)
resultats_csv.to_csv(sortie, index=False)
print(f"R?sultats sauvegard?s dans : {sortie}")